# Text Classification from Scratch

Importing Libraries

In [ ]:
import tensorflow as tf
import numpy as np

Loading the dataset

In [ ]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  51.9M      0  0:00:01  0:00:01 --:--:-- 51.8M


Preparing train, validation, and test dataset

In [ ]:
batch_size = 32
raw_train_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size = batch_size,
    validation_split = 0.2,
    subset = "training",
    seed = 1337,
)

raw_validation_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size = batch_size,
    validation_split= 0.2,
    subset = "validation",
    seed = 1337,
)

raw_test_ds = tf.keras.utils.text_dataset_from_directory(
    "aclImdb/test",
    batch_size = batch_size,
)

Found 75000 files belonging to 3 classes.
Using 60000 files for training.
Found 75000 files belonging to 3 classes.
Using 15000 files for validation.
Found 25000 files belonging to 2 classes.


Vectorizing Data

In [ ]:
from tensorflow.keras.layers import TextVectorization
import string
import re

# raw text consists of html tags
# hence we need to remove these tags using custom standardization
def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  html_stripped = tf.strings.regex_replace(lowercase, "<br />", " ")
  return tf.strings.regex_replace(html_stripped, f"[{re.escape(string.punctuation)}]", "")

# model constants
max_features = 20000
embedding_dim = 128
sequence_length = 500

# vectorization
vectorize_layer = TextVectorization(  
    standardize = custom_standardization,
    max_tokens = max_features,
    output_mode = "int",
    output_sequence_length = sequence_length,
  )

# creating vocabulary
text_ds = raw_train_ds.map(lambda x, y: x)

# adapting to the dataset
vectorize_layer.adapt(text_ds)

# vectorization
def vectorize_text(text, label):
  text = tf.expand_dims(text, 1)
  return vectorize_layer(text), label

# vectorizing train data
train_ds = raw_train_ds.map(vectorize_text)
val_ds = raw_train_ds.map(vectorize_text)
test_ds = raw_test_ds.map(vectorize_text)

Build Model

In [ ]:
from tensorflow.keras import layers

# input layer
inputs = tf.keras.Input(shape = (None, ), dtype = "int64")

# embedding
x = layers.Embedding(max_features, embedding_dim)(inputs)
x = layers.Dropout(0.5)(x)

# convolution layers
x = layers.Conv1D(128, 7, padding = "valid", activation = "relu", strides = 3)(x)
x = layers.Conv1D(128, 7, padding = "valid", activation = "relu", strides = 3)(x)
x = layers.GlobalMaxPooling1D()(x)

# hidden vanilla layer
x = layers.Dense(128, activation = "relu")(x)
x = layers.Dropout(0.5)(x)

# output layer
predictions = layers.Dense(1, activation = "sigmoid", name = "predictions")(x)

# model
model = tf.keras.Model(inputs, predictions)

# compiling the model
model.compile(loss = "binary_crossentropy", optimizer = "adam", metrics = ["accuracy"])

Training the model

In [ ]:
epochs = 3

model.fit(train_ds, validation_data = val_ds, epochs = epochs)

Epoch 1/3
1875/1875 [==============================] - 437s 232ms/step - loss: -374310862848.0000 - accuracy: 0.1664 - val_loss: -1997437009920.0000 - val_accuracy: 0.1664
Epoch 2/3
1875/1875 [==============================] - 450s 240ms/step - loss: -14254627356672.0000 - accuracy: 0.1664 - val_loss: -35868878307328.0000 - val_accuracy: 0.1664
Epoch 3/3
1875/1875 [==============================] - 439s 234ms/step - loss: -95475646595072.0000 - accuracy: 0.1664 - val_loss: -173945588285440.0000 - val_accuracy: 0.1664


Evaluating the model

In [ ]:
model.evaluate(test_ds)

782/782 [==============================] - 33s 42ms/step - loss: 175764171390976.0000 - accuracy: 0.5000


[175764171390976.0, 0.5]

# Text Classification using Transformer

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Transformer Block

In [ ]:
from keras.backend_config import epsilon
class TransformerBlock(layers.Layer):
  def __init__(self, embed_dim, num_heads, ff_dim, rate = 0.1):
    super().__init__()
    self.att = layers.MultiHeadAttention(num_heads = num_heads, key_dim = embed_dim)
    self.ffn = keras.Sequential(
        [layers.Dense(ff_dim, activation = "relu"),
         layers.Dense(embed_dim),]
    )
    self.layernorm1 = layers.LayerNormalization(epsilon = 1e-7)
    self.layernorm2 = layers.LayerNormalization(epsilon = 1e-7)
    self.dropout1 = layers.Dropout(rate)
    self.dropout2 = layers.Dropout(rate)

  def call(self, inputs, training):
    attn_output = self.att(inputs, inputs)
    attn_output = self.dropout1(attn_output, training = training)
    out1 = self.layernorm1(inputs + attn_output)
    ffn_output = self.ffn(out1)
    ffn_output = self.dropout2(ffn_output, training = training)
    return self.layernorm2(out1 + ffn_output)

Embedding Layer

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
  def __init__(self, maxlen, vocab_size, embed_dim):
    super().__init__()
    self.token_emb = layers.Embedding(input_dim = vocab_size, output_dim = embed_dim)
    self.pos_emb = layers.Embedding(input_dim = maxlen, output_dim = embed_dim)

  def call(self, x):
    maxlen = tf.shape(x)[-1]
    positions = tf.range(start=0, limit=maxlen, delta=1)
    positions = self.pos_emb(positions)
    x = self.token_emb(x)
    return x + positions

Preparing Dataset

In [ ]:
vocab_size = 20000
maxlen = 200

(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words = vocab_size)

x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen) 

17464789/17464789 [==============================] - 0s 0us/step


Building Classifier

In [ ]:
embed_dim = 32
num_heads = 2
ff_dim = 32

inputs = layers.Input(shape = (maxlen, ))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation = "relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(2, activation = "softmax")(x)

model = keras.Model(inputs = inputs, outputs = outputs)

Training the model

In [ ]:
model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])
history = model.fit(x_train, y_train, batch_size = 32, epochs = 2, validation_data = (x_val, y_val))

Epoch 1/2
782/782 [==============================] - 13s 14ms/step - loss: 0.4539 - accuracy: 0.7951 - val_loss: 0.3212 - val_accuracy: 0.8618
Epoch 2/2
782/782 [==============================] - 11s 14ms/step - loss: 0.2365 - accuracy: 0.9072 - val_loss: 0.3026 - val_accuracy: 0.8754


# Text Classification with RNN

In [ ]:
import numpy
import tensorflow as tf
import tensorflow_datasets as tfds

Data Preprocessing

In [ ]:
dataset, info = tfds.load('imdb_reviews', with_info = True, as_supervised = True)
train_ds, test_ds = dataset['train'], dataset['test']

# shuffling data for training and creating batches
BUFFER_SIZE = 10000
BATCH_SIZE = 64

train_dataset = train_ds.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_ds.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

Creating Encoder

In [ ]:
VOCAB_SIZE = 1000
encoder = tf.keras.layers.TextVectorization(max_tokens = VOCAB_SIZE)
encoder.adapt(train_dataset.map(lambda text, label: text))

Creating the Model

In [ ]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(input_dim = len(encoder.get_vocabulary()), output_dim = 64, mask_zero = True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(1)
])

Compiling the Model

In [ ]:
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(from_logits = True),
    optimizer = tf.keras.optimizers.Adam(1e-4),
    metrics = ['accuracy']
)

Training the Model

In [ ]:
history = model.fit(train_dataset, epochs = 10, validation_data = test_dataset, validation_steps = 30)

Epoch 1/10
391/391 [==============================] - 745s 2s/step - loss: 0.6371 - accuracy: 0.5679 - val_loss: 0.4591 - val_accuracy: 0.7802
Epoch 2/10
391/391 [==============================] - 717s 2s/step - loss: 0.3908 - accuracy: 0.8241 - val_loss: 0.3461 - val_accuracy: 0.8323
Epoch 3/10
391/391 [==============================] - 694s 2s/step - loss: 0.3381 - accuracy: 0.8506 - val_loss: 0.3689 - val_accuracy: 0.8016
Epoch 4/10
391/391 [==============================] - 733s 2s/step - loss: 0.3261 - accuracy: 0.8563 - val_loss: 0.3253 - val_accuracy: 0.8557
Epoch 5/10
391/391 [==============================] - 701s 2s/step - loss: 0.3116 - accuracy: 0.8641 - val_loss: 0.3407 - val_accuracy: 0.8464
Epoch 6/10
391/391 [==============================] - 697s 2s/step - loss: 0.3058 - accuracy: 0.8683 - val_loss: 0.3249 - val_accuracy: 0.8661
Epoch 7/10
391/391 [==============================] - 684s 2s/step - loss: 0.3040 - accuracy: 0.8669 - val_loss: 0.3073 - val_accuracy: 0.8714

Model Evaluation

In [ ]:
model.evaluate(test_dataset)